# Parameter Fraction Analysis

Analyzes the effect of computing the Jacobian w.r.t. a random subset of parameters
on SVD optimizer performance across all datasets. At each step, a random `param_fraction`
of parameters is selected and only those columns of the Jacobian are computed (coordinate-descent style).

## 1. Setup & Data Loading

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from style import set_style, lr_labels
set_style()

PLOT_DIR = Path('plots/paramfrac')
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {PLOT_DIR.resolve()}")

In [ ]:
# Load results
from style import load_results

SCAN_CONFIGS = {
    'CIFAR-10 (CE)':       'cifar10_resnet_paramfrac_scan',
    'CIFAR-10 (LabelReg)': 'cifar10_resnet_paramfrac_labelreg_scan',
    'MNIST (CE)':          'mnist_paramfrac_ce_scan',
    'MNIST (LabelReg)':    'mnist_paramfrac_labelreg_scan',
    'Toy 1D':              'toy_1d_paramfrac_scan',
    'Polynomial':          'polynomial_paramfrac_scan',
}

datasets = {}
for name, scan_name in SCAN_CONFIGS.items():
    try:
        df = load_results(scan_name)
        datasets[name] = df
    except FileNotFoundError:
        print(f"  [skip] {name}: not found")

print(f"\nLoaded {len(datasets)} datasets: {list(datasets.keys())}")


In [ ]:
# Helper functions
def get_loss_curve(row, loss_type='val'):
    return np.array(row['losses'][loss_type])

def get_acc_curve(row, acc_type='val_acc'):
    return np.array(row['losses'].get(acc_type, []))

def sliding_average(data, window=10):
    return np.convolve(data, np.ones(window)/window, mode='valid')

def add_derived_columns(df):
    df = df.copy()
    df['final_val_loss'] = df.apply(lambda r: r['losses']['val'][-1], axis=1)
    df['final_train_loss'] = df.apply(lambda r: r['losses']['train'][-1], axis=1)
    df['final_val_acc'] = df.apply(lambda r: r['losses'].get('val_acc', [np.nan])[-1], axis=1)
    df['final_train_acc'] = df.apply(lambda r: r['losses'].get('train_acc', [np.nan])[-1], axis=1)
    df['total_time'] = df['losses'].apply(lambda l: l.get('total_time', np.nan))
    df['avg_batch_time_train'] = df['losses'].apply(lambda l: l.get('avg_batch_time_train', np.nan))
    # Ensure param_fraction is numeric
    df['param_fraction'] = df['param_fraction'].fillna(1.0).astype(float)
    return df

for name in datasets:
    datasets[name] = add_derived_columns(datasets[name])

for name, df in datasets.items():
    pf_vals = sorted(df['param_fraction'].unique())
    print(f"{name}: {len(df)} runs, param_fractions={pf_vals}")

## 2. Best Final Loss vs Parameter Fraction

For each param_fraction, find the best SVD config (across lr, k, rtol) and compare.

In [ ]:
colors = sns.color_palette('deep')
has_acc = any(not np.isnan(df['final_val_acc'].iloc[0]) for df in datasets.values())
ncols = 2 if has_acc else 1

fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 6))
if ncols == 1:
    axes = [axes]

for i, (name, df) in enumerate(datasets.items()):
    pf_vals = sorted(df['param_fraction'].unique())
    best_losses = [df[df['param_fraction'] == pf]['final_val_loss'].min() for pf in pf_vals]
    best_accs = [df[df['param_fraction'] == pf]['final_val_acc'].max() for pf in pf_vals]

    c = colors[i % len(colors)]
    axes[0].plot(pf_vals, best_losses, 'o-', color=c, lw=2.5, markersize=7, label=name)

    if ncols > 1:
        valid = [(pf, a) for pf, a in zip(pf_vals, best_accs) if not np.isnan(a)]
        if valid:
            axes[1].plot([v[0] for v in valid], [v[1]*100 for v in valid],
                         'o-', color=c, lw=2.5, markersize=7, label=name)

axes[0].set_xlabel('Parameter Fraction')
axes[0].set_ylabel('Best Final Validation Loss')
axes[0].set_title('Best SVD Performance vs Parameter Fraction')
axes[0].set_yscale('log')
axes[0].legend(frameon=False, fontsize=10)

if ncols > 1:
    axes[1].set_xlabel('Parameter Fraction')
    axes[1].set_ylabel('Best Final Validation Accuracy (%)')
    axes[1].set_title('Best SVD Accuracy vs Parameter Fraction')
    axes[1].legend(frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'best_loss_vs_paramfrac.pdf')
plt.show()

## 3. Training Curves by Parameter Fraction

Best SVD training curve at each param_fraction, per dataset.

In [ ]:
n_datasets = len(datasets)
fig, axes = plt.subplots(1, n_datasets, figsize=(6 * n_datasets, 5), squeeze=False)
axes = axes[0]

cmap = plt.cm.viridis

for col, (name, df) in enumerate(datasets.items()):
    ax = axes[col]
    pf_vals = sorted(df['param_fraction'].unique())
    n_pf = len(pf_vals)
    pf_colors = [cmap(j / max(n_pf - 1, 1)) for j in range(n_pf)]

    for j, pf in enumerate(pf_vals):
        pf_df = df[df['param_fraction'] == pf]
        best_idx = pf_df['final_val_loss'].idxmin()
        row = df.loc[best_idx]
        val_curve = get_loss_curve(row, 'val')
        n_params_eff = int(pf * 100)
        ax.plot(range(len(val_curve)), val_curve, color=pf_colors[j], lw=2,
                label=f'pf={pf:.0%}')

    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation Loss')
    ax.set_yscale('log')
    ax.set_title(name)
    ax.legend(fontsize=9, frameon=False, loc='upper right')

plt.suptitle('Best SVD Training Curves by Parameter Fraction', y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'training_curves_by_paramfrac.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Accuracy curves (classification datasets only)
class_datasets = {k: v for k, v in datasets.items()
                  if not np.isnan(v['final_val_acc'].iloc[0])}

if class_datasets:
    n_cd = len(class_datasets)
    fig, axes = plt.subplots(1, n_cd, figsize=(6 * n_cd, 5), squeeze=False)
    axes = axes[0]

    for col, (name, df) in enumerate(class_datasets.items()):
        ax = axes[col]
        pf_vals = sorted(df['param_fraction'].unique())
        n_pf = len(pf_vals)
        pf_colors = [cmap(j / max(n_pf - 1, 1)) for j in range(n_pf)]

        for j, pf in enumerate(pf_vals):
            pf_df = df[df['param_fraction'] == pf]
            best_idx = pf_df['final_val_acc'].idxmax()
            row = df.loc[best_idx]
            val_acc = get_acc_curve(row, 'val_acc')
            if len(val_acc) > 0:
                ax.plot(range(1, len(val_acc)+1), val_acc * 100,
                        color=pf_colors[j], lw=2, label=f'pf={pf:.0%}')

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Validation Accuracy (%)')
        ax.set_title(name)
        ax.legend(fontsize=9, frameon=False, loc='lower right')

    plt.suptitle('Best SVD Accuracy Curves by Parameter Fraction', y=1.02)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'accuracy_curves_by_paramfrac.pdf', bbox_inches='tight')
    plt.show()

## 4. Wall Time vs Parameter Fraction

Smaller param_fraction means a narrower Jacobian, so each step should be cheaper.
But the update only moves a subset of parameters, so more steps may be needed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for i, (name, df) in enumerate(datasets.items()):
    pf_vals = sorted(df['param_fraction'].unique())
    best_times = []
    best_batch_times = []
    for pf in pf_vals:
        pf_df = df[df['param_fraction'] == pf]
        best_idx = pf_df['final_val_loss'].idxmin()
        best_times.append(df.loc[best_idx, 'total_time'])
        best_batch_times.append(df.loc[best_idx, 'avg_batch_time_train'])

    c = colors[i % len(colors)]
    axes[0].plot(pf_vals, best_times, 'o-', color=c, lw=2.5, markersize=7, label=name)
    axes[1].plot(pf_vals, best_batch_times, 'o-', color=c, lw=2.5, markersize=7, label=name)

for ax, ylabel, title in zip(axes,
        ['Total Training Time (s)', 'Avg Batch Time (s)'],
        ['Total Time (Best Config)', 'Per-Batch Time (Best Config)']):
    ax.set_xlabel('Parameter Fraction')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_yscale('log')
    ax.legend(frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'wall_time_vs_paramfrac.pdf')
plt.show()

## 5. Efficiency Frontier: Loss vs Wall Time

In [ ]:
n_datasets = len(datasets)
fig, axes = plt.subplots(1, n_datasets, figsize=(6 * n_datasets, 5), squeeze=False)
axes = axes[0]

for col, (name, df) in enumerate(datasets.items()):
    ax = axes[col]
    pf_vals = sorted(df['param_fraction'].unique())
    n_pf = len(pf_vals)
    pf_colors = [cmap(j / max(n_pf - 1, 1)) for j in range(n_pf)]

    for j, pf in enumerate(pf_vals):
        pf_df = df[df['param_fraction'] == pf]
        ax.scatter(pf_df['total_time'], pf_df['final_val_loss'],
                   color=pf_colors[j], alpha=0.6, s=40, label=f'pf={pf:.0%}')

    ax.set_xlabel('Total Time (s)')
    ax.set_ylabel('Final Val Loss')
    ax.set_yscale('log')
    ax.set_title(name)
    ax.legend(fontsize=8, frameon=False, loc='upper right')

plt.suptitle('Efficiency Frontier: Loss vs Wall Time', y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'efficiency_frontier_paramfrac.pdf', bbox_inches='tight')
plt.show()

## 6. Heatmaps: k × Parameter Fraction

Best final val loss (across lr, seeds) as a function of k and param_fraction.

In [ ]:
n_datasets = len(datasets)
fig, axes = plt.subplots(1, n_datasets, figsize=(5 * n_datasets, 4), squeeze=False)
axes = axes[0]

for col, (name, df) in enumerate(datasets.items()):
    ax = axes[col]

    pivot = df.pivot_table(
        values='final_val_loss',
        index='k',
        columns='param_fraction',
        aggfunc='min'
    )

    if pivot.empty:
        ax.set_title(f'{name}\n(no data)')
        continue

    im = ax.imshow(np.log10(pivot.values), aspect='auto', cmap='viridis')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'{pf:.0%}' for pf in pivot.columns], rotation=45)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([str(int(k)) for k in pivot.index])
    ax.set_xlabel('Param Fraction')
    ax.set_ylabel('k')
    ax.set_title(name)

    for yi in range(len(pivot.index)):
        for xi in range(len(pivot.columns)):
            val = pivot.values[yi, xi]
            if not np.isnan(val):
                ax.text(xi, yi, f'{val:.3f}', ha='center', va='center',
                        fontsize=7, color='white' if np.log10(val) < np.log10(pivot.values[~np.isnan(pivot.values)]).mean() else 'black')

fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('log$_{10}$(Val Loss)')

plt.suptitle('Best Val Loss: k × Parameter Fraction', y=1.02)
plt.savefig(PLOT_DIR / 'heatmap_k_vs_paramfrac.pdf', bbox_inches='tight')
plt.show()

## 7. Singular Value Analysis by Parameter Fraction

Does restricting to a parameter subset change the effective rank of the Jacobian?

In [ ]:
n_datasets = len(datasets)
fig, axes = plt.subplots(1, n_datasets, figsize=(6 * n_datasets, 5), squeeze=False)
axes = axes[0]

for col, (name, df) in enumerate(datasets.items()):
    ax = axes[col]
    bs = df['batch_size'].iloc[0]
    pf_vals = sorted(df['param_fraction'].unique())
    n_pf = len(pf_vals)
    pf_colors = [cmap(j / max(n_pf - 1, 1)) for j in range(n_pf)]

    for j, pf in enumerate(pf_vals):
        pf_df = df[df['param_fraction'] == pf]
        best_idx = pf_df['final_val_loss'].idxmin()
        row = df.loc[best_idx]

        svd_info = row.get('svd_info')
        if svd_info is None or not isinstance(svd_info, dict):
            continue
        num_nonzero = svd_info.get('num_nonzero_svs', [])
        if len(num_nonzero) == 0:
            continue

        n_epoch = len(row['losses']['train'])
        num_nonzero = np.array(num_nonzero)
        svs_norm = num_nonzero / bs
        x = np.linspace(0, n_epoch, len(svs_norm))

        sw = max(1, len(svs_norm) // (n_epoch * 3))
        if sw > 1:
            svs_smooth = sliding_average(svs_norm, window=sw)
            ax.plot(x[sw-1:], svs_smooth, color=pf_colors[j], lw=2,
                    label=f'pf={pf:.0%}')
        else:
            ax.plot(x, svs_norm, color=pf_colors[j], lw=2,
                    label=f'pf={pf:.0%}')

    ax.set_xlabel('Epoch')
    ax.set_ylabel('Active SVs / Batch Size')
    ax.set_ylim(0, 1.15)
    ax.set_title(name)
    ax.legend(fontsize=9, frameon=False)

plt.suptitle('Effective Rank Evolution by Parameter Fraction', y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'sv_rank_by_paramfrac.pdf', bbox_inches='tight')
plt.show()

## 8. Cross-Dataset Comparison: Normalized Loss Decay

In [ ]:
colors_ds = sns.color_palette('deep')
linestyles = ['-', '--', ':', '-.', (0, (3, 1, 1, 1))]

# One subplot per param_fraction value (pick shared ones)
all_pfs = set()
for df in datasets.values():
    all_pfs.update(df['param_fraction'].unique())
shared_pfs = sorted(all_pfs)

# Show extremes + middle
show_pfs = [shared_pfs[0], shared_pfs[len(shared_pfs)//2], shared_pfs[-1]]
show_pfs = sorted(set(show_pfs))  # deduplicate

fig, axes = plt.subplots(1, len(show_pfs), figsize=(6 * len(show_pfs), 5))
if len(show_pfs) == 1:
    axes = [axes]

for ax_idx, pf in enumerate(show_pfs):
    ax = axes[ax_idx]
    for i, (name, df) in enumerate(datasets.items()):
        pf_df = df[df['param_fraction'] == pf]
        if len(pf_df) == 0:
            continue
        best_idx = pf_df['final_val_loss'].idxmin()
        row = df.loc[best_idx]
        val_curve = get_loss_curve(row, 'val')
        val_norm = val_curve / val_curve[0]
        x = np.linspace(0, 1, len(val_norm))
        ax.plot(x, val_norm, color=colors_ds[i % len(colors_ds)], lw=3, label=name)

    ax.set_xlabel('Training Progress')
    ax.set_ylabel('Val Loss / Initial')
    ax.set_yscale('log')
    ax.set_title(f'pf = {pf:.0%}')
    ax.legend(frameon=False, fontsize=9)

plt.suptitle('Normalized Loss Decay by Parameter Fraction', y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'normalized_loss_decay_paramfrac.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Per-dataset: all param_fractions, normalized loss
n_datasets = len(datasets)
fig, axes = plt.subplots(1, n_datasets, figsize=(6 * n_datasets, 5), squeeze=False)
axes = axes[0]

for col, (name, df) in enumerate(datasets.items()):
    ax = axes[col]
    pf_vals = sorted(df['param_fraction'].unique())
    n_pf = len(pf_vals)
    pf_colors = [cmap(j / max(n_pf - 1, 1)) for j in range(n_pf)]

    for j, pf in enumerate(pf_vals):
        pf_df = df[df['param_fraction'] == pf]
        best_idx = pf_df['final_val_loss'].idxmin()
        row = df.loc[best_idx]
        val_curve = get_loss_curve(row, 'val')
        val_norm = val_curve / val_curve[0]
        ax.plot(range(len(val_norm)), val_norm, color=pf_colors[j], lw=2,
                label=f'pf={pf:.0%}')

    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val Loss / Initial')
    ax.set_yscale('log')
    ax.set_title(name)
    ax.legend(fontsize=9, frameon=False)

plt.suptitle('Normalized Val Loss by Parameter Fraction', y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'normalized_loss_all_paramfrac.pdf', bbox_inches='tight')
plt.show()

## 9. Speedup vs Quality Tradeoff

Plot the relative speedup (batch time at pf=X / batch time at pf=1.0) against
relative loss degradation.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for i, (name, df) in enumerate(datasets.items()):
    pf_vals = sorted(df['param_fraction'].unique())
    if 1.0 not in pf_vals:
        continue

    # Reference: best run at pf=1.0
    ref_df = df[df['param_fraction'] == 1.0]
    ref_loss = ref_df['final_val_loss'].min()
    ref_time = ref_df.loc[ref_df['final_val_loss'].idxmin(), 'avg_batch_time_train']

    speedups = []
    loss_ratios = []
    pf_labels = []
    for pf in pf_vals:
        pf_df = df[df['param_fraction'] == pf]
        best_idx = pf_df['final_val_loss'].idxmin()
        best_loss = pf_df['final_val_loss'].min()
        best_time = df.loc[best_idx, 'avg_batch_time_train']
        speedups.append(ref_time / best_time if best_time > 0 else 1.0)
        loss_ratios.append(best_loss / ref_loss)
        pf_labels.append(pf)

    c = colors[i % len(colors)]
    ax.plot(speedups, loss_ratios, 'o-', color=c, lw=2.5, markersize=8, label=name)
    for s, l, pf in zip(speedups, loss_ratios, pf_labels):
        ax.annotate(f'{pf:.0%}', (s, l), textcoords='offset points',
                    xytext=(5, 5), fontsize=8, color=c)

ax.axhline(1.0, color='gray', ls='--', lw=1, alpha=0.5)
ax.axvline(1.0, color='gray', ls='--', lw=1, alpha=0.5)
ax.set_xlabel('Batch Time Speedup (relative to pf=100%)')
ax.set_ylabel('Loss Ratio (relative to pf=100%)')
ax.set_title('Speedup vs Quality Tradeoff')
ax.legend(frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'speedup_vs_quality_paramfrac.pdf')
plt.show()

## 10. Summary Table

In [ ]:
summary_rows = []
for name, df in datasets.items():
    for pf in sorted(df['param_fraction'].unique()):
        pf_df = df[df['param_fraction'] == pf]
        best_idx = pf_df['final_val_loss'].idxmin()
        row = df.loc[best_idx]
        summary_rows.append({
            'Dataset': name,
            'pf': f'{pf:.0%}',
            'Best Val Loss': f"{row['final_val_loss']:.4f}",
            'Val Acc': f"{row['final_val_acc']*100:.1f}%" if not np.isnan(row['final_val_acc']) else '\u2014',
            'lr': row['lr'],
            'k': int(row['k']) if not np.isnan(row.get('k', np.nan)) else '\u2014',
            'Batch Time (s)': f"{row['avg_batch_time_train']:.4f}",
            'Total Time (s)': f"{row['total_time']:.1f}",
        })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))